# BGE-M3 → Qdrant (full-length legal articles)

Mỗi Điều luật là một point. Dùng pooling chính thức của `SentenceTransformer`, L2 normalization, cosine distance và tối đa 8192 token. Collection tương ứng với fusion: `laws_bge_m3_v2_correct_pooling`.

In [2]:
!pip -q install -U sentence-transformers qdrant-client
import os, gc, json, glob
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from kaggle_secrets import UserSecretsClient
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

MODEL_ID = "BAAI/bge-m3"
COLLECTION = "laws_bge_m3_v2_correct_pooling"
MAX_LENGTH = 8192
ENCODE_BATCH = 1
UPSERT_BATCH = 128
EXPECTED_DIM = 1024

assert torch.cuda.is_available(), "Hãy bật GPU Kaggle."
secrets = UserSecretsClient()
qc = QdrantClient(url=secrets.get_secret("QDRANT_URL"),
                  api_key=secrets.get_secret("QDRANT_KEY"), timeout=300)
print("GPU:", torch.cuda.get_device_name(0))
print("Qdrant collections:", [x.name for x in qc.get_collections().collections])

GPU: Tesla T4
Qdrant collections: ['alqac_laws_raw', 'laws_gte_qwen2', 'alqac_laws_test']


In [3]:
def find_corpus():
    paths = ["/kaggle/working/corpus_law_pub.json", "corpus_law_pub.json"]
    paths += glob.glob("/kaggle/input/**/corpus_law_pub.json", recursive=True)
    for path in paths:
        if os.path.exists(path): return path
    raise FileNotFoundError("Không thấy corpus_law_pub.json")

corpus_path = find_corpus()
corpus = json.load(open(corpus_path, encoding="utf-8"))
records = []
for law in corpus:
    for article_no, article in enumerate(law["content"], 1):
        records.append({"point_id": len(records), "aid": int(article["aid"]),
                        "law_id": law["law_id"], "article_no": article_no,
                        "text": article["content_Article"]})
print("Corpus:", corpus_path, "| laws:", len(corpus), "| articles:", len(records))
assert len(records) == 3352

Corpus: /kaggle/input/datasets/ldhhieu18/corpus-law/corpus_law_pub.json | laws: 18 | articles: 3352


In [4]:
model = SentenceTransformer(MODEL_ID, device="cuda")
model.max_seq_length = MAX_LENGTH
tokenizer = model.tokenizer

# Thống kê chính xác và dừng nếu vẫn có Điều bị cắt.
token_lengths = []
for i, rec in enumerate(records, 1):
    n = len(tokenizer(rec["text"], add_special_tokens=True, truncation=False)["input_ids"])
    token_lengths.append(n)
    if i % 500 == 0 or i == len(records): print(" token-count", i, "/", len(records))
too_long = [(records[i]["law_id"], records[i]["article_no"], records[i]["aid"], n)
            for i, n in enumerate(token_lengths) if n > MAX_LENGTH]
print("Token length min/mean/max:", min(token_lengths), float(np.mean(token_lengths)), max(token_lengths))
print("Articles exceeding", MAX_LENGTH, ":", len(too_long))
if too_long: print(sorted(too_long, key=lambda x: -x[3])[:20])
assert not too_long, "Có Điều vượt 8192 token; không được tiếp tục embedding."

vectors = model.encode([r["text"] for r in records], batch_size=ENCODE_BATCH,
                       normalize_embeddings=True, show_progress_bar=True,
                       convert_to_numpy=True)
assert vectors.shape == (len(records), EXPECTED_DIM)
assert np.isfinite(vectors).all()
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-3)
print("Vectors:", vectors.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

 token-count 500 / 3352
 token-count 1000 / 3352
 token-count 1500 / 3352
 token-count 2000 / 3352
 token-count 2500 / 3352
 token-count 3000 / 3352
 token-count 3352 / 3352
Token length min/mean/max: 16 219.8019093078759 2781
Articles exceeding 8192 : 0


Batches:   0%|          | 0/3352 [00:00<?, ?it/s]

Vectors: (3352, 1024)


In [5]:
# Chỉ thay collection sau khi toàn bộ vector đã tạo và kiểm tra thành công.
if qc.collection_exists(COLLECTION): qc.delete_collection(COLLECTION)
qc.create_collection(COLLECTION, vectors_config=VectorParams(size=EXPECTED_DIM, distance=Distance.COSINE))
for start in range(0, len(records), UPSERT_BATCH):
    points = [PointStruct(id=r["point_id"], vector=v.tolist(), payload={
        "aid": r["aid"], "law_id": r["law_id"], "article_no": r["article_no"],
        "content_Article": r["text"], "embedding_model": MODEL_ID,
        "embedding_pipeline": "sentence_transformer_official_pooling_l2",
        "max_length": MAX_LENGTH,
    }) for r, v in zip(records[start:start+UPSERT_BATCH], vectors[start:start+UPSERT_BATCH])]
    qc.upsert(COLLECTION, points=points, wait=True)
    print(" upserted", min(start + UPSERT_BATCH, len(records)), "/", len(records))
count = int(qc.count(COLLECTION).count)
print("✓", COLLECTION, "dim=", EXPECTED_DIM, "points=", count)
assert count == len(records)

# Document alignment.
for idx in [0, len(records)//4, len(records)//2, 3*len(records)//4, len(records)-1]:
    hits = qc.query_points(COLLECTION, query=vectors[idx].tolist(), limit=3, with_payload=True).points
    aids = [int(h.payload["aid"]) for h in hits]
    assert records[idx]["aid"] in aids
    print(" alignment aid=", records[idx]["aid"], "rank=", aids.index(records[idx]["aid"])+1)
print("✓ Document/collection alignment passed")

 upserted 128 / 3352
 upserted 256 / 3352
 upserted 384 / 3352
 upserted 512 / 3352
 upserted 640 / 3352
 upserted 768 / 3352
 upserted 896 / 3352
 upserted 1024 / 3352
 upserted 1152 / 3352
 upserted 1280 / 3352
 upserted 1408 / 3352
 upserted 1536 / 3352
 upserted 1664 / 3352
 upserted 1792 / 3352
 upserted 1920 / 3352
 upserted 2048 / 3352
 upserted 2176 / 3352
 upserted 2304 / 3352
 upserted 2432 / 3352
 upserted 2560 / 3352
 upserted 2688 / 3352
 upserted 2816 / 3352
 upserted 2944 / 3352
 upserted 3072 / 3352
 upserted 3200 / 3352
 upserted 3328 / 3352
 upserted 3352 / 3352
✓ laws_bge_m3_v2_correct_pooling dim= 1024 points= 3352
 alignment aid= 270 rank= 1
 alignment aid= 50824 rank= 2
 alignment aid= 53219 rank= 1
 alignment aid= 56697 rank= 1
 alignment aid= 57130 rank= 1
✓ Document/collection alignment passed


In [6]:
def search(query, top=5):
    qv = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    hits = qc.query_points(COLLECTION, query=qv.tolist(), limit=top, with_payload=True).points
    for rank, hit in enumerate(hits, 1):
        p = hit.payload or {}
        print(rank, round(hit.score, 4), p.get("law_id"), "Điều", p.get("article_no"), "aid", p.get("aid"))
    return hits
search("bồi thường thiệt hại do súc vật gây ra")

1 0.6485 91/2015/QH13 Điều 603 aid 53373
2 0.5682 45/2013/QH13 Điều 90 aid 56040
3 0.5631 91/2015/QH13 Điều 604 aid 53374
4 0.5466 100/2015/QH13 Điều 241 aid 56685
5 0.5426 91/2015/QH13 Điều 608 aid 53378


[ScoredPoint(id=1830, version=15, score=0.6485292, payload={'aid': 53373, 'law_id': '91/2015/QH13', 'article_no': 603, 'content_Article': '1. Chủ sở hữu súc vật phải bồi thường thiệt hại do súc vật gây ra cho người khác. Người chiếm hữu, sử dụng súc vật phải bồi thường thiệt hại trong thời gian chiếm hữu, sử dụng súc vật, trừ trường hợp có thỏa thuận khác.\n\n2. Trường hợp người thứ ba hoàn toàn có lỗi làm cho súc vật gây thiệt hại cho người khác thì người thứ ba phải bồi thường thiệt hại; nếu người thứ ba và chủ sở hữu cùng có lỗi thì phải liên đới bồi thường thiệt hại.\n\n3. Trường hợp súc vật bị chiếm hữu, sử dụng trái pháp luật gây thiệt hại thì người chiếm hữu, sử dụng trái pháp luật phải bồi thường; khi chủ sở hữu, người chiếm hữu, sử dụng súc vật có lỗi trong việc để súc vật bị chiếm hữu, sử dụng trái pháp luật thì phải liên đới bồi thường thiệt hại.\n\n4. Trường hợp súc vật thả rông theo tập quán mà gây thiệt hại thì chủ sở hữu súc vật đó phải bồi thường theo tập quán nhưng khô